In [ ]:
import re

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import anndata as ad
import gseapy as gp
import scanpy as sc

matplotlib.rcParams['font.size'] = 16.0

%matplotlib inline

from eykthyr.eykthyr import Eykthyr, load_anndata

os.makedirs('figure_panels', exist_ok=True)
sc.settings.figdir = 'figure_panels'


# Panel A

In [ ]:
# Load dataset
adata = sc.read_h5ad(
    "publication_figure_data/slidetags/slidetags_processed.h5ad"
)

datasets = [adata]

# Leiden cluster ID -> label
leiden_labels = {
    "0": "CD8+ T cell",
    "1": "Tumor 1",
    "2": "Mono-mac",
    "3": "Tumor 2",
    "4": "Plasma",
    "5": "Proliferating T cell",
    "6": "T-reg",
    "7": "NK cell",
    "8": "Cytotoxic CD8+ T cell",
    "9": "mDC",
}

# Final plotting order
label_order = [
    "Tumor 1",
    "Tumor 2",
    "Proliferating T cell",
    "CD8+ T cell",
    "Cytotoxic CD8+ T cell",
    "Regulatory T cell",
    "mDC",
    "Mono-mac",
    "NK cell",
    "Plasma",
]

# Label -> color
label_colors = {
    "Tumor 1": "#e34a33",
    "Tumor 2": "#feb24c",
    "Proliferating T cell": "#8856a7",
    "CD8+ T cell": "#deebf7",
    "Cytotoxic CD8+ T cell": "#7fcdbb",
    "Regulatory T cell": "#a1d99b",
    "mDC": "#756bb1",
    "Mono-mac": "#c994c7",
    "NK cell": "#bcbddc",
    "Plasma": "#a6bddb",
}

In [ ]:
def add_labeled_leiden(
    adata,
    source_col="original_leiden",
    target_col="labeled_leiden",
    cluster_to_label=None,
    label_order=None,
    label_colors=None,
):
    """Map Leiden cluster IDs to labels, set category order, and assign Scanpy colors."""

    cluster_to_label = cluster_to_label or {}
    label_order = label_order or []
    label_colors = label_colors or {}

    # Map cluster IDs to labels
    adata.obs[target_col] = (
        adata.obs[source_col]
        .astype(str)
        .map(cluster_to_label)
        .replace({"T-reg": "Regulatory T cell"})
    )

    # Check for unmapped clusters
    if adata.obs[target_col].isna().any():
        unmapped = adata.obs.loc[adata.obs[target_col].isna(), source_col].unique()
        raise ValueError(f"Unmapped Leiden clusters found: {unmapped}")

    # Keep only labels present in this AnnData, preserving desired order
    present_labels = set(adata.obs[target_col].unique())
    actual_categories = [label for label in label_order if label in present_labels]

    # Check for labels missing from label_order
    unexpected_labels = present_labels - set(label_order)
    if unexpected_labels:
        raise ValueError(f"Labels missing from label_order: {unexpected_labels}")

    # Check for missing colors
    missing_colors = set(actual_categories) - set(label_colors)
    if missing_colors:
        raise ValueError(f"Labels missing from label_colors: {missing_colors}")

    # Set categorical order
    adata.obs[target_col] = pd.Categorical(
        adata.obs[target_col],
        categories=actual_categories,
        ordered=True,
    )

    # Assign colors in category order
    adata.uns[f"{target_col}_colors"] = [
        label_colors[label] for label in actual_categories
    ]

    return adata

In [ ]:
# Apply to all datasets
datasets = [
    add_labeled_leiden(
        d,
        source_col="original_leiden",
        target_col="labeled_leiden",
        cluster_to_label=leiden_labels,
        label_order=label_order,
        label_colors=label_colors,
    )
    for d in datasets
]

# Store in Eykthyr object
e = Eykthyr()
e.perturbed_X = datasets

# Plot
sc.pl.spatial(
    e.perturbed_X[0],
    color="labeled_leiden",
    spot_size=60,
    frameon=False,
    title="",
    save="Panel_4A.svg",
)

# Panel B

In [ ]:
Tcells = [e.perturbed_X[0][(e.perturbed_X[0].obs['original_leiden'] == '0') | (e.perturbed_X[0].
                     obs['original_leiden'] == '5') | (e.perturbed_X[0].
                     obs['original_leiden'] == '7') | (e.perturbed_X[0].
                     obs['original_leiden'] == '6') | (e.perturbed_X[0].
                    obs['original_leiden'] == '8')].copy()]

Tcells = [Tcells[0][Tcells[0].obs['cell_type'].isin(['T_CD4', 'T_CD8', 'T_reg'])].copy()]
Tcells[0].obs['cell_type'] = Tcells[0].obs['cell_type'].replace({'T_reg': 'Regulatory T cell'})
Tcells[0].obs['cell_type'] = Tcells[0].obs['cell_type'].replace({'T_CD4': 'CD4 T cell'})
Tcells[0].obs['cell_type'] = Tcells[0].obs['cell_type'].replace({'T_CD8': 'CD8 T cell'})

sc.pp.neighbors(Tcells[0], use_rep='normalized_X')
sc.tl.umap(Tcells[0])

sc.tl.paga(Tcells[0], groups='cell_type')

leiden_labels = {'0': 'CD8+ T cell',
                 '1': 'Tumor 1',
                 '2': 'Mono-mac',
                 '3': 'Tumor 2',
                 '4': 'Plasma',
                 '5': 'Proliferating T cell',
                 '6': 'Regulatory T cell',
                 '7': 'NK cell',
                 '8': 'Cytotoxic CD8+ T cell',
                 '9': 'mDC'
                }

Tcells[0].obs['new_labels'] = Tcells[0].obs['original_leiden'].map(leiden_labels)

original_leiden_palette = {
    'CD8+ T cell': '#a8ddb5',
    'Proliferating T cell': '#fa9fb5',
    'NK cell': '#8856a7',
    'Regulatory T cell': '#2c7fb8',
    'Cytotoxic CD8+ T cell': '#c994c7'
}


cell_type_palette = {
    'CD4 T cell': '#3182bd',       
    'CD8 T cell': '#fdae6b',
    'Regulatory T cell': '#31a354'
}

sc.pl.umap(Tcells[0], color=['new_labels'], palette=original_leiden_palette, title='', size=100,  save='Panel4B_top.svg', edgecolor=None, frameon=False,legend_loc=None)

sc.pl.umap(Tcells[0], color=['cell_type'], palette=cell_type_palette, size=100, title='', save='Panel4B_bot.svg', edgecolor=None, frameon=False,legend_loc=None)



# Panel C

In [ ]:
top_tf = ['POU3F3',
 'SP2',
 'E2F6',
 'ZNF281',
 'POU2F1',
 'HES1',
 'KLF15',
 'WT1',
 'KLF5',
 'PURA',
 'TFAP2D',
 'SP1',
 'ZNF148',
 'KLF6',
 'ZNF263',
 'DNMT1',
 'MAZ',
 'EGR1',
 'ZBTB7A',
 'CTCFL']

bot_tf = ['PURA',
 'ZFX',
 'KLF16',
 'SP2',
 'YY1',
 'KLF4',
 'KLF6',
 'TFAP2D',
 'DNMT1',
 'ZBTB7A',
 'ZNF281',
 'KLF15',
 'ZNF148',
 'SP1',
 'ZNF263',
 'E2F6',
 'MAZ',
 'KLF5',
 'CTCFL',
 'EGR1']

In [ ]:
# Differential TFs

rdf_top = pd.read_csv('publication_figure_data/slidetags/benchmarker_cluster0-5_top.csv')
rdf_top = rdf_top.set_index('Embedding')
rdf_top = rdf_top.drop(index=['Metric Type'])

rdf_bot = pd.read_csv('publication_figure_data/slidetags/benchmarker_cluster0-5_bottom.csv')
rdf_bot = rdf_bot.set_index('Embedding')
rdf_bot = rdf_bot.drop(index=['Metric Type'])

In [ ]:
rdf_bot['Batch correction'] = pd.to_numeric(rdf_bot['Batch correction'], errors='coerce')
rdf_top['Batch correction'] = pd.to_numeric(rdf_top['Batch correction'], errors='coerce')

rdf_top = rdf_top.sort_values(by='Batch correction', ascending=False)[:30]
rdf_bot = rdf_bot.sort_values(by='Batch correction', ascending=False)[:30]

difference_df = pd.DataFrame({
    'Batch correction difference': np.abs(rdf_top['Batch correction'] - rdf_bot['Batch correction'])
})

# Drop NaN values if there are any embeddings that are not present in both DataFrames
difference_df.dropna(inplace=True)

# Sort the DataFrame by the difference
sorted_difference_df = difference_df.sort_values(by='Batch correction difference', ascending=False)
sorted_difference_df

In [ ]:
sorted_difference_df.index = sorted_difference_df.index.str.replace('^normalized_X_', '').str.replace('_dropout$', '')


In [ ]:
sorted_difference_df = sorted_difference_df[:10]

In [ ]:
plt.rcParams["font.size"] = 16
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 16
plt.rcParams["xtick.labelsize"] = 13
plt.rcParams["ytick.labelsize"] = 14

def clean_tf_label(label):
    label = str(label).strip()

    # Standardize a few messy variants
    label = label.replace("normalized X", "normalized_X")
    label = label.replace(" dropout", "_dropout")
    label = label.replace("-", "")
    label = label.replace("*", "")

    # Extract TF name from patterns like normalized_X_KLF5_dropout
    match = re.search(r"normalized_X_(.*?)_dropout", label)
    if match:
        return match.group(1)

    return label

values = sorted_difference_df["Batch correction difference"].to_numpy()
labels = [clean_tf_label(x) for x in sorted_difference_df.index]
x = np.arange(len(values))

# TFs to highlight in red
highlight_tfs = {"KLF4", "YY1"}

colors = ["#d73027" if tf in highlight_tfs else "#2b8c99" for tf in labels]

fig, ax = plt.subplots(figsize=(4.3, 3.2))

# Draw stems manually
for xi, yi, c in zip(x, values, colors):
    ax.vlines(xi, 0, yi, color=c, linewidth=1.2)
    ax.scatter(xi, yi, color=c, s=55, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90, ha="center")

# Give a tiny bit of headroom so top dots are not clipped,
# while still labeling the axis up to 0.3
ax.set_ylim(0, 0.32)
ax.set_yticks([0.0, 0.1, 0.2, 0.3])

ax.set_ylabel("Tumor score\nvariation")
ax.set_title("TFs with differential impact", pad=6)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

ax.margins(x=0.05)

fig.subplots_adjust(left=0.22, bottom=0.34, top=0.82, right=0.97)

plt.savefig(
    "figure_panels/Panel_4C.svg",
    format="svg",
    bbox_inches="tight"
)

plt.show()

# Panel D

In [ ]:
Tcells_X = []
for d in Tcells:
    d_X = sc.AnnData(X=d.obsm['X'], obs=d.obs, obsm=d.obsm)
    Tcells_X.append(d_X)

Tcells_X[0].obsm['spatial_scaled'] = (Tcells_X[0].obsm['spatial'] - Tcells_X[0].obsm['spatial'].
                                        min()) / Tcells_X[0].obsm['spatial'].max() * 100

for d in Tcells_X:
    d.obsm['spatial_2'] = d.obsm['spatial_scaled'].copy()
    d.obsm['spatial_2'][:,1] = d.obsm['spatial_2'][:,1] * -1

ymean = Tcells_X[0].obsm['spatial_scaled'][:,1].mean()
Tcells_X[0].obs['top_half'] = ['no' if Tcells_X[0].obsm['spatial_scaled'][t,1] > ymean else 'yes' for t in range(len(Tcells_X[0].obs_names))] 

top = Tcells_X[0][Tcells_X[0].obs['top_half']=='yes']
bottom = Tcells_X[0][Tcells_X[0].obs['top_half']=='no']

In [ ]:
concat = ad.concat([top, bottom])

In [ ]:
adata_subset = e.perturbed_X[0][e.perturbed_X[0].obs['labeled_leiden'].isin(['CD8+ T cell', 'Cytotoxic CD8+ T cell', 'NK cell',
       'Proliferating T cell', 'T-reg'])].copy()


In [ ]:
import gseapy as gp
from gseapy import barplot, dotplot
import pandas as pd
from mpl_toolkits.axes_grid1 import make_axes_locatable

def get_response_dfs(tf, datasets, datasets_X, datasetnames):
    # Calculate global absmax
    global_absmax = 0
    response_dfs = [pd.DataFrame(index=d.var.index, columns=d.obs['labeled_leiden'].unique()) for d in datasets]
    for d, df in zip(datasets_X, response_dfs):
        d.obsm[f'{tf}_mse'] = d.obsm['X'] - d.obsm[f'X_{tf}_dropout']
        ct_df = pd.DataFrame(index = [f'X_{i}' for i in range(16)], columns = [ct for ct in d.obs['labeled_leiden'].unique()])
        for ct in d.obs['labeled_leiden'].unique():
            ct_mse = d[d.obs['labeled_leiden'] == ct].obsm[f'{tf}_mse'].mean(axis=0)
            ct_df.loc[:,ct] = ct_mse
            gene_ct_mse = np.matmul(datasets[0].uns['M'][datasetnames[0]],ct_mse)
            df.loc[:, ct] = gene_ct_mse
        absmax = max(abs(ct_df.to_numpy().min()), abs(ct_df.to_numpy().max()))
        global_absmax = max(global_absmax, absmax)

    for d, df in zip(datasets_X, response_dfs):
        d.obsm[f'{tf}_mse'] = d.obsm['X'] - d.obsm[f'X_{tf}_dropout']
        ct_df = pd.DataFrame(index = [f'X_{i}' for i in range(16)], columns = [ct for ct in d.obs['labeled_leiden'].unique()])
        for ct in d.obs['labeled_leiden'].unique():
            ct_mse = d[d.obs['labeled_leiden'] == ct].obsm[f'{tf}_mse'].mean(axis=0)
            ct_df.loc[:,ct] = ct_mse
            gene_ct_mse = np.matmul(datasets[0].uns['M'][datasetnames[0]],ct_mse)
            df.loc[:, ct] = gene_ct_mse

        fig, ax = plt.subplots(1,1,figsize=(8,8))
        ct_df = ct_df.apply(pd.to_numeric, errors="coerce")
        heatmap_values = ct_df.to_numpy(dtype=float)
        
        im = ax.imshow(
            heatmap_values.T,
            cmap="bwr",
            vmin=-global_absmax,
            vmax=global_absmax,
        )        
        ax.set_yticks(np.arange(len(ct_df.columns)), labels=ct_df.columns)
        ax.set_xticks(np.arange(len(ct_df.index)), labels=[f'{k}' for k in range(len(ct_df.index))])
        
        # Create a divider for the existing axes instance
        divider = make_axes_locatable(ax)
        # Append axes to the right of ax, with 5% width of ax
        cax = divider.append_axes("right", size="5%", pad=0.1)
        # Create colorbar in the appended axes
        cbar = plt.colorbar(im, cax=cax)

        plt.savefig('figure_panels/Panel_4D.svg', bbox_inches='tight')
        plt.show()

    return response_dfs

ans_top = get_response_dfs('YY1', [adata_subset], [top], ['slidetags'])

#labels for top and bottom half, like "Proliferating T cell" becomes "Proliferating T cell - top" and "Proliferating T cell - bottom"

In [ ]:
def get_response_dfs(tf, datasets, datasets_X, datasetnames, common_labels):
    response_dfs = [pd.DataFrame(index=d.var.index, columns=common_labels) for d in datasets]
    for d, df in zip(datasets_X, response_dfs):
        d.obsm[f'{tf}_mse'] = d.obsm['X'] - d.obsm[f'X_{tf}_dropout']
        ct_df = pd.DataFrame(index=[f'X_{i}' for i in range(16)], columns=common_labels)
        for ct in common_labels:
            if ct in d.obs['labeled_leiden'].unique():
                ct_mse = d[d.obs['labeled_leiden'] == ct].obsm[f'{tf}_mse'].mean(axis=0)
                ct_df.loc[:, ct] = ct_mse
                gene_ct_mse = np.matmul(datasets[0].uns['M'][datasetnames[0]], ct_mse)
                df.loc[:, ct] = gene_ct_mse
        absmax = max(abs(ct_df.to_numpy().min()), abs(ct_df.to_numpy().max()))
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        ax.set_yticks(np.arange(len(ct_df.columns)), labels=ct_df.columns)
        ax.set_xticks(np.arange(len(ct_df.index)), labels=[f'{k}' for k in range(len(ct_df.index))])
        ct_df = ct_df.apply(pd.to_numeric, errors="coerce")
        heatmap_values = ct_df.to_numpy(dtype=float)
        
        absmax = np.nanmax(np.abs(heatmap_values))
        
        im = ax.imshow(
            heatmap_values.T,
            cmap="bwr",
            vmin=-absmax,
            vmax=absmax,
        )
        
        print(ct_df.to_numpy().min(), ct_df.to_numpy().max())
        plt.savefig(f'figure_panels/{tf}_mchange_{datasetnames[0]}.svg', bbox_inches='tight')
        plt.show()
    return response_dfs

# Get the common labels
common_labels = list(set(top.obs['labeled_leiden'].unique()).union(set(bottom.obs['labeled_leiden'].unique())))

# Update labels for top and bottom datasets to ensure unique labels
# top.obs['labeled_leiden'] = top.obs['labeled_leiden'] + ' - top'
# bottom.obs['labeled_leiden'] = bottom.obs['labeled_leiden'] + ' - bottom'

# Call the function with the updated datasets and common labels
ans_top = get_response_dfs('YY1', [adata_subset], [top], ['slidetags'], common_labels)
ans_bottom = get_response_dfs('YY1', [adata_subset], [bottom], ['slidetags'], common_labels)



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def get_response_dfs(tf, datasets, datasets_X, datasetnames, common_labels):
    # Gene-level response dfs
    response_dfs = [
        pd.DataFrame(np.nan, index=d.var.index, columns=common_labels, dtype=float)
        for d in datasets
    ]
    
    ct_dfs = []

    for d, df, dataset, datasetname in zip(datasets_X, response_dfs, datasets, datasetnames):
        # Compute TF response in latent space
        d.obsm[f"{tf}_mse"] = d.obsm["X"] - d.obsm[f"X_{tf}_dropout"]

        # Latent-dimension x cell-type dataframe
        n_latent = d.obsm["X"].shape[1]
        ct_df = pd.DataFrame(
            np.nan,
            index=[f"X_{i}" for i in range(n_latent)],
            columns=common_labels,
            dtype=float,
        )

        for ct in common_labels:
            if ct in d.obs["labeled_leiden"].unique():
                mask = d.obs["labeled_leiden"] == ct

                # Mean latent response for this cell type
                ct_mse = np.asarray(
                    d[mask].obsm[f"{tf}_mse"].mean(axis=0),
                    dtype=float
                ).ravel()

                ct_df.loc[:, ct] = ct_mse

                # Project latent response back to gene space
                gene_ct_mse = np.asarray(
                    dataset.uns["M"][datasetname] @ ct_mse,
                    dtype=float
                ).ravel()

                df.loc[:, ct] = gene_ct_mse

        ct_dfs.append(ct_df)

        # ---- Plot latent response heatmap ----
        heatmap_values = ct_df.to_numpy(dtype=float).T
        absmax = np.nanmax(np.abs(heatmap_values))
        if not np.isfinite(absmax) or absmax == 0:
            absmax = 1.0

        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        im = ax.imshow(
            heatmap_values,
            cmap="bwr",
            vmin=-absmax,
            vmax=absmax,
            aspect="auto"
        )

        ax.set_yticks(np.arange(len(ct_df.columns)))
        ax.set_yticklabels(ct_df.columns)

        ax.set_xticks(np.arange(len(ct_df.index)))
        ax.set_xticklabels([f"{k}" for k in range(len(ct_df.index))])

        print("ct_df range:", np.nanmin(heatmap_values), np.nanmax(heatmap_values))

        plt.colorbar(im, ax=ax)
        plt.savefig(f"figure_panels/{tf}_mchange_{datasetname}.svg", bbox_inches="tight")
        plt.show()

    return response_dfs, ct_dfs


def plot_difference_heatmap(top, bottom, tf, common_labels):
    n_latent = top.obsm["X"].shape[1]

    ct_df_top = pd.DataFrame(
        np.nan,
        index=[f"X_{i}" for i in range(n_latent)],
        columns=common_labels,
        dtype=float,
    )
    ct_df_bottom = pd.DataFrame(
        np.nan,
        index=[f"X_{i}" for i in range(n_latent)],
        columns=common_labels,
        dtype=float,
    )

    for ct in common_labels:
        if ct in top.obs["labeled_leiden"].unique():
            mask_top = top.obs["labeled_leiden"] == ct
            ct_mse_top = np.asarray(
                top[mask_top].obsm[f"{tf}_mse"].mean(axis=0),
                dtype=float
            ).ravel()
            ct_df_top.loc[:, ct] = ct_mse_top

        if ct in bottom.obs["labeled_leiden"].unique():
            mask_bottom = bottom.obs["labeled_leiden"] == ct
            ct_mse_bottom = np.asarray(
                bottom[mask_bottom].obsm[f"{tf}_mse"].mean(axis=0),
                dtype=float
            ).ravel()
            ct_df_bottom.loc[:, ct] = ct_mse_bottom

    diff_df = ct_df_top - ct_df_bottom
    heatmap_values = diff_df.to_numpy(dtype=float).T

    absmax = np.nanmax(np.abs(heatmap_values))
    if not np.isfinite(absmax) or absmax == 0:
        absmax = 1.0

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    im = ax.imshow(
        heatmap_values,
        cmap="bwr",
        vmin=-absmax,
        vmax=absmax,
        aspect="equal"
    )

    ax.set_yticks([])
    ax.set_xticks([])

    print("diff_df range:", np.nanmin(heatmap_values), np.nanmax(heatmap_values))

    plt.colorbar(im, ax=ax)
    plt.savefig(f"figure_panels/{tf}_mchange_difference.svg", bbox_inches="tight")
    plt.show()


# Get the common labels
common_labels = sorted(
    set(top.obs["labeled_leiden"].unique()).union(
        set(bottom.obs["labeled_leiden"].unique())
    )
)

# Plot difference heatmap
plot_difference_heatmap(top, bottom, "YY1", common_labels)

# Panel E

In [ ]:
genes_top_higher = [g.strip() for g in open('publication_figure_data/slidetags/genes_top_higher_YY1_openness_10kb.txt','r').readlines()]
genes_bot_higher = [g.strip() for g in open('publication_figure_data/slidetags/genes_bot_higher_YY1_openness_10kb.txt','r').readlines()]
background = [g.strip() for g in open('publication_figure_data/slidetags/YY1_openness_background_10kb.txt','r').readlines()]

In [ ]:
k = 200
topktop = genes_top_higher[:k]
topkbot = genes_bot_higher[:k]

In [ ]:
enr = gp.enrichr(gene_list=topkbot,
                 gene_sets=['GO_Biological_Process_2023','Reactome_2022'],
                 organism='human', 
                 background=background,
                 outdir=None,
                )
enr.results.sort_values(by='Adjusted P-value')

In [ ]:
dpallmenr = enr #Put the cell type index from enrs that you want the GSEA dot plot for

#comment out below 
GOenr = dpallmenr.results[dpallmenr.results['Gene_set'] == 'GO_Biological_Process_2023'].sort_values(by='Adjusted P-value')
##
reactomeenr = dpallmenr.results[dpallmenr.results['Gene_set'] == 'Reactome_2022'].sort_values(by='Adjusted P-value')

X2 = reactomeenr['Term'].values[:5].tolist()
X2 = [' '.join(x.split()[:-1]) for x in X2]
X2.reverse()
X2

In [ ]:
height2 = reactomeenr['Adjusted P-value'].values[:5].tolist()
height2 = [np.log10(1 / float(h)) for h in height2]
height2.reverse()

In [ ]:
#exclude GO processes
marker_sizes2 = reactomeenr['Odds Ratio'].values[:5].tolist()
marker_sizes2.reverse()
desired_max = 20
desired_min = 5
#make min be 0
min_marker_size = min(marker_sizes2)
max_marker_size = max(marker_sizes2)
print(f'min Odds Ratio: {min_marker_size}\nmax Odds Ratio: {max_marker_size}')
marker_sizes2 = [(m - min_marker_size) + desired_min  for m in marker_sizes2]
#scale between 0,(desired_max - desired_min)
max_marker_size = max(marker_sizes2)
marker_sizes2 = [(m / max_marker_size) * (desired_max - desired_min) for m in marker_sizes2]
#add desired_min to now scale between desired_min,desired_max
marker_sizes2 = [m+ desired_min  for m in marker_sizes2]
print(marker_sizes2)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,5))
for ms, x, h in zip(marker_sizes2, X2, height2):
    markerline2, stemlines2, baseline2 = ax.stem(x,h,orientation='horizontal', markerfmt='C0o', basefmt='w-', 
                                             label='Reactome', linefmt='#7570b3')
    markerline2.set_markerfacecolor('#7570b3')
    markerline2.set_markeredgecolor('#7570b3')
    markerline2.set_markersize(ms)

ax.set_xlabel(r'$-\log$(adj. p-val.)')

plt.savefig('figure_panels/Panel_4E.svg', format='svg', bbox_inches='tight')


# ax.legend(loc='upper right')